In [59]:
from os import listdir, makedirs
from os.path import isfile, isdir, join

import numpy as np
import pandas as pd

flare_file_path = './output/non_ar_matched.csv'
headers_path = '../sharps-headers/'
output_merged_sharp_path = '../annotated-new-data/'

sharp_last_date = pd.to_datetime('2017-06-30', format='%Y-%m-%d')

noaa_map_path = './all_harps_with_noaa_ars.txt'
noaa_map = False  # global variable
downloads_noaa_map = False # global variable

from flare_reader import get_flare_dataframe, match_harp_to_noaa, get_harp_to_noaa_map
from aia_flare_integration import get_aia_flare_dataframe, get_goes_flare_dataframe

noaa_map = get_harp_to_noaa_map()
ufdf = get_flare_dataframe(flare_file_path)

# ufdf['end_time'] < sharp_last_date


In [60]:
ufdf = ufdf[ ufdf['end_time'] < sharp_last_date]
# print ufdf.shape[0]


In [61]:
cols = ['goes_location', 'aia_location', 'noaa_active_region_aia', 'noaa_active_region_goes'] 
aia_only = ufdf[(pd.isnull(ufdf['noaa_active_region_goes']))]
both = ufdf[~(pd.isnull(ufdf['noaa_active_region_goes']))]
goes_only = ufdf[(pd.isnull(ufdf['noaa_active_region_aia']))]
both = both[~(pd.isnull(both['noaa_active_region_aia']))]

In [4]:
foi_aia = aia_only[ aia_only['noaa_active_region_aia']!=0 ]
# foi_aia.head()
foi_aia.shape[0]

37

In [5]:
foi_goes = goes_only[goes_only['noaa_active_region_goes']!=0]
# foi_goes.head()
foi_goes.shape[0]

247

In [6]:
foi_both = both[ ~((both['noaa_active_region_aia'] == 0) & (both['noaa_active_region_goes'] == 0)) ]
foi_both.shape[0]

640

In [7]:
foi = pd.concat( [foi_both, foi_goes, foi_aia] )
foi.shape[0]

924

In [62]:
def get_noaa_to_harp_map(noaa_map):
    noaa_to_harp = {}
    for harp,noaa_list in noaa_map.to_dict()['NOAA_ARS'].iteritems():
        for noaa in noaa_list.split(','):
            harps = noaa_to_harp.get(noaa, [])
            harps.append(harp)
            noaa_to_harp[noaa] = harps
    return noaa_to_harp

noaa_to_harps = get_noaa_to_harp_map(noaa_map)


In [63]:
aia_harps = ufdf['noaa_active_region_aia'].apply( lambda x: noaa_to_harps.get(str(int(x)), []) if ~np.isnan(x) and x!=0 else [])
goes_harps = ufdf['noaa_active_region_goes'].apply( lambda x: noaa_to_harps.get(str(int(x)), []) if ~np.isnan(x) and x!=0 else [])
harps = pd.DataFrame({'aia': aia_harps2,'goes':goes_harps2})



,aia,goes
flare_id,,
69,[86],[]
70,[86],[]
293,[],[]
350,[],[]
683,[],[]
684,[],[]
1776,[1621],[]
1777,[1621],[1621]
1778,[1621],[]


In [65]:
harp_lists = harps.apply( lambda x : list(set(x['aia'] + x['goes'])), axis='columns' )
ufdf['harps'] = harp_lists
cols.append('harps')


,goes_location,aia_location,noaa_active_region_aia,noaa_active_region_goes,harps
flare_id,,,,,
69,"(0, 0)",POINT(62 12),11087.0,11091.0,[86]
70,"(0, 0)",POINT(61 12),11087.0,11091.0,[86]
293,"(0, 0)",POINT(62 -24),0.0,0.0,[]
350,"(0, 0)",POINT(-62 32),0.0,0.0,[]
683,"(0, 0)",POINT(-70 8),0.0,0.0,[]
684,"(0, 0)",POINT(-8 22),0.0,0.0,[]
1776,"(0, 0)",POINT(-64 -23),11471.0,0.0,[1621]
1777,"(-63, -24)",POINT(-63 -23),11471.0,11471.0,[1621]
1778,"(0, 0)",POINT(-62 -23),11471.0,0.0,[1621]


In [66]:
ufdf.to_csv('./output/unmatched_flares_with_harps.csv')

In [69]:
all_harps = []
for harp_list in ufdf['harps'].values:
    print harp_list
    all_harps.extend(harp_list)
all_harps = list( set( all_harps ) )
with open('./output/all_harps_combined', 'w') as file_handler:
    for item in all_harps:
        file_handler.write("{}\n".format(item))

[86]
[86]
[]
[]
[]
[]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1621]
[1677]
[1677]
[1727]
[1750]
[1750]
[1750]
[1750]
[1744]
[1750]
[1750]
[1750]
[1750]
[1750]
[1744]
[1750]
[1750]
[1750]
[1744]
[1750]
[1750]
[1750]
[1795, 1789]
[1789]
[1795]
[1795]
[1795]
[1795]
[1795]
[1806]
[1806]
[1795, 1806]
[1795]
[1806]
[1795]
[1806]
[1795]
[1806]
[1795]
[1806]
[1806]
[1806]
[1806]
[1806]
[1807]
[1795]
[1806]
[1807]
[1807]
[1806]
[1806]
[1795]
[1806]
[1807]
[1807]
[1806]
[1807]
[1806]
[1806]
[1806]
[1806]
[1807]
[1806, 1807]
[1807]
[1807]
[1807]
[1806]
[1806]
[1806]
[1807]
[1806]
[1807]
[1807]
[1807]
[1807]
[1807]
[1806]
[1807]
[1807]
[1806]
[1807]
[1806]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1806]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1806]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[1807]
[

In [75]:
import os.path
for harp in all_harps:
    print os.path.isfile('../sharps-headers/' + str(harp) + '.txt')
    print '\t',os.path.isfile('../new-data/' + str(harp) + '.csv')

True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	True
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
True
	False
